# GRU - Mạng  Neural hồi tiếp với nút có cổng

## 1. Mô hình ngôn ngữ

Dữ liệu chuối là dạng dữ liệu mang có ý nghĩa và mang tính chất tuần tự, như: Âm nhạc, giọng nói, vnaw bản, phim ảnh, bước đi,... Nếu chúng ta hoán vị chúng, chúng sẽ không còn mang nhiều ý nghĩa, ví dụ như tiêu đề "Vợ chồng tỷ phú Bill Gates vừa ly hôn sau gầnb 30 năm bên nhau" thì mang nhiều ý nghĩa hơn là tiêu đề "Ly hôn tỷ phú vợ chồng Bill Gates sau gần 30 năm bên nhau"

Dữ liệu dạng văn bản là 1 ví dụ điển hình về dữ liệu chuỗi. Mỗi bài post trên facebook là một chuỗi các từ, cũng là chuỗi các ký tự. Dữ liệu văn bản là dạng dữ liệu quan trọng cuối cùng với dữ liệu hình ảnh trong lĩnh vực học máy.

Việc tiền dữ liệu văn bản gồm 4 bước:
- Nạp dữ liệu văn bản ở dạng chuỗi vào bộ nhớ.
- Chia chuỗi vừa nạp thành các token, mỗi token là 1 từ hoặc 1 ký tự.
- Xây dựng bộ từ vựng để ánh xạ các token thành chỉ số để phân biệt chúng với nhau (token_to_idx).
- Ánh xạ tất cả token trong văn bản thành các chỉ số tương ứng để dễ dàng đưa vào mô hình.

Mình có 1 văn bản có độ dài là $T$, mỗi ký tự là 1 token, nên văn bản là 1 chuỗi các quan sát (các số) rời rạc. Giả sử văn bản trên có dãy token là $x_1,x_2,x_3,...,x_T$ với $x_T$ (1 $\leq$ t $\leq$ $T$) được coi là đầu ra tại bước thời gain $t$, khi đã có chuỗi thời gian trên, mục tiêu của mô hình phải tính được xác suât của:
$$
p(x_1,x_2,...,x_T)
$$
Một mô hình ngôn ngữ lý tưởng có thể tự tạo ra văn bản tự nhiên bằng việc chọn $w_t$ ở bước thời gian $t$ với $x_T$ $∼$ $p(w_t | w_{t-1}, ..., w_1)$

Vậy làm thế nào để mô hình hoá một tài liệu hay thậm chí là 1 chuỗi các từ?

Chúng ta sẽ áp dụng quy tắc xác xuất cơ bản sau:

$$
p(Statistics, is, func, ....) = p(Statistics).p(is | Statistics).p(fun | Statistics).p(. | Statistics,is,fun)
$$

Mình cùng nhớ lại mô hình $Markov$ và áp dụng để mô hình hoá ngôn ngữ. Một phân phối trên các chuỗi thoả mãn điều kiện Markov bậc một nếu:
$$
p(w_{t+1} | w_t,...,w_1) = p(w_{w+1} | w_t)
$$

Các bậc cao hơn ứng với các chuỗi phụ thuộc dài hơn. Do đó, chúng ta có thể áp dụng xấp xỉ:
$$
p(w_1,w_2,w_3,w_4) = p(w_1).p(w_2).p(w_3).p(w_4)
$$
$$
p(w_1,w_2,w_3,w_4) = p(w_1).p(w_2|w_1).p(w_3|w_2).p(w_4|w_3)
$$
$$
p(w_1,w_2,w_3,w_4) = p(w_1).p(w_2|w_1).p(w_3|w_1,w_2).p(w_4|w_2,w_3)
$$

Các công thức xác suất trên lần lượt được gọi là unigram, bigram và trigram. Các công thức này đều có dạng n-gram.

## 2. Mạng neural hồi tiếp

Như mô hình n-gram mình vừa tìm hiểu phía trên, xác suất có điều kiện của từ $x_t$ tại vị trí $t$ chỉ phụ thuộc vào $n-1$ từ trước đó. Rõ ràng là muốn kiểm tra xem 1 từ ở vị trí phía trước vị trí $t - (n-1)$, ta sẽ phỉa tăng n lên theo, đồng nghĩa với số tham số mô hình sẽ tăng theo hàm mũ vì ta cần lưu $|V|^n$ giá trị với 1 từ điển $V$ nào đó. Do đó, sẽ tốt hơn nếu chúng ta dùng mô hình biến tiềm ẩn:
$$
p(x_t | x_{t-1},...,x_1) ≈ p(x_t | x_{t-1},h_t)
$$

$h_t$ được gọi là trạng thái ẩn, để luuwu các thông tin của chuỗi cho đến thời điểm hiện tại. Trạng thái ẩn $h_t$ được tính bằng cả $x_t$ và trạng thái ẩn trước đó $h_{h-1}$:
$$
h_t = f(x_t, h_{t-1})
$$

Việc dùng thêm trạng thái ẩn có thể khiến việc tính toán và lưu trữ của mô hình trở nên nặng nề.

![](image1.png)

Ở đâu, $t$ đưcoj gọi là bước thời gian. Với mỗi $t$ ta có $X_t \in \mathbb{R}^{n\times d}$ và $H_t \in \mathbb{R}^{n\times h}$ là trạng thái ẩn ở bước thời gian $t$ của chuỗi. Ở đây ta dùng thêm $W_{hh} \in \mathbb{R}^{h\times h}$ để làm tham số mô tả cho việc dùng trạng thái ẩn trước đó cho dự đoán ở bước thời gian hiện tại:
$$
H_t = ϕ(X_t.W_{xh} + H_{t-1}.W_{hh} + b_h)
$$

Chúng ta có đầu ra khá giống với perceptron đa tầng:
$$
O_t = H_t.W_{hq} + b_q
$$

Ở đây sau khi kết nối đầu vào $X_t$ với trạng thái ẩn trước đó $H_{t-1}$, ta coi nó như 1 input đầu vào của 1 tầng kết nối đầy đủ với hàm kích hoạt ϕ, đầu ra là trạng thái ẩn ở bước thời gian hiện tại $H_t$. $H_t$ đưcoj dùng để tính $H_{t+1}$ là trạng thái ẩn ở bước thời gian tiếp theo, đồng thời được dùng để tính giá trị đầu ra ở bước thời gian hiện tại.

## 3. Mạng hồi tiếp nút có cổng

Từ công thức phần 2, ta rút ra:
$$
h_t = f(x_t, h_{t-1}, w_h)
$$
$$
o_t = g(h_t, w_o)
$$

Ta có chuỗi các giá trị {..., $(h_{t-1},x_{t-1},o_{t-1})$,$(h_t, x_t, o_t)$} phụ thuộc nhau và có tính chất đệ quy. Vì tính chất này, với nhiều bước thời gian thì có thể gây ra hiệnt ượng tiêu biến hoặc bùng nổ gradient.

Ta sẽ gặp các tình huống như sau:
- Ta gặp 1 quan sát xuất hiện sớm và ảnh hưởng rất lớn đến toàn bộ các quan sát phía sau. Thường thì ta phải gán 1 giá trị cực lớn cho gradient của quan sát phan đầu đó, nhưng ta có thể dùng 1 cơ chế để lưu thông tin quan trọng ở quan sát ban đầu vào ô nhớ.

- Tình huống khác là các quan sát phía trước không mang nhiều ý nghĩa để phục vụ cho việc dự đoán các quan sát phía sau, như khi phân tích 1 trang HTML ta có thể gặp thẻ `<mark>` nhưng nó không giúp gì cho việc truyền tải thông tin. Do đó ta muốn bỏ qua những ký tự như vậy trong các biểu diễn trạng thái ẩn.

- Với các văn bản có các chương, khi xuống dòng chuyển qua chương mới thì ta đặt lại các trạng thái ẩn về ban đầu, bới hầu như ý nghĩa của chương phía sau không liên quan đén chương phía trước.

Có rất nhiều ý tưởng để giải quyết các vấn đề trên, một trong những phương pháp ra đời sớm nấht là Bộ nhớ ngắn hạn($LSTM$), nút hồi tiếp có cổng $(GRU)$ là 1 biến thể khác của LSTM, thường có chất lượng tương đương nhưng tốc độ tính toán nhanh hơn đáng kể.

Khác biệt chính giữa $RNN$ thông thường và $GRU$ là Gru cho phép điều khiển trạng thái ẩn, tức là ta có các cơ chế học để xem khi nào cập nhật và khi nào nên xoá trạng thái ẩn. Ví dụ như với các quan sát quan trọng, mô hình sẽ học để giữ nguyên trạng thái ẩn của quan sát đó. Với những quan sát không liên quan, mô hình sẽ xoá bỏ các trạng thái ẩn đó khi cần thiết.

### Cổng xoá và cổng cập nhật

Giả sử ta có biến xoá và biến cập nhật, biến xoá cho phép kiểm soát bao nhiêu phần mà trạng thái trước đây được giữ lại, biến cập nhật cho phép kiểm soát trạng thái ẩn mới có bao nhiêu phần giống trạng thái ẩn cũ.

Ta sẽ đi thiết kế các cổng cho các biến đó, với đầu vào ở bước thời gian hiện tại là $X_t$ và trạng thái ẩn ở bước trước đó là $H_{t-1}$, ta sẽ có 2 biến đại diện cho 2 cổng:
- cổng xoá: $R_t \in \mathbb{R}^{n \times h}$
- cổng cập nhật: $Z_t \in \mathbb{R}^{n \times h}$

$$
R_t = \sigma(X_t.W_{xr} + H_{t-1}.W_{hr} + b_r)
$$
$$
Z_t = \sigma(X_t.W_{xz} + H_{t-1}.W_{hz} + b_z)
$$

![](image2.png)

Trong đó, $W_{xr}$, $W_{xz}$ $\in \mathbb{R}^{d \times h}$ và $W_{hr}$, $W_{hz}$, $\in \mathbb{R}^{h \times h}$ là các trọng số và $b_1,b_2 \in \mathbb{R}^{1 \times h}$ là các tham số độ chênh. Dùng hàm $sigmoid$ để 2 giá trị thu được $\in (0,1)$.

#### + Hoạt động của cổng xoá

Quay trở lại với công thức thông thường của RNN:
$$
H_t = Tanh(X_t.W_{xh} + H_{t-1}.W_{hh} + b_h)
$$

Với hàm kích hoạt là hàm $Tanh$ để giá trị $\in (-1, 1)$

![](image3.png)

Để giảm ảnh hưởng của trạng thái ẩn trước đó, ta có công thức sau:
$$
\~{H_t} = Tanh(X_t.W_{xh} + (R_t ⊙ H_{t-1}).W_{hh} + b_h)
$$

Ta thấy $R_t$ gần $0$ thì trạng thái ẩn đầu ra chính là output của multiperceptron 1 tầng với input là $X_t$ và các trạng thái ẩn trước đó đều đạt về mặc định, nên $\~{H_t}$ được gọi là trạng thái ẩn tiềm năng. Ngược lại nếu gần $1$, thì công thức lại quay trở về thông thường.

#### + Hoạt động của cổng cập nhật

![](image4.png)

Cổng cập nhật xác định mức giống nhau giữa trạng thái ẩn hiện tại $H_t$ và $H_{t-1}$:
$$
H_t = Z_t ⊙ H_{t-1} + (1 - Z_t) ⊙ \~{H_t}
$$

Nếu giá trị của $Z_t = 1$ thì $H_t = H_{t-1}$. Trong trường hợp này, thông tin của $X_t$ sẽ bị bỏ qua, tương đương với việc bỏ qua bước thời gian $t$ trong chuỗi thời gian. Ngược lại, nếu $Z_t = 0$ thì trạng thái ẩn $H_t$ sẽ gần giống với trạng thái ẩn tiềm năng $\~{H_t}$.

Những thiết kế trên có thể giúp mô hình RNN giải quyết vấn đề triệt tiêu hoặc bùng ổn gradient và nắm bắt tốt hơn các thông tin của các quan sát trong chuỗi thời gian.